# SmartBite PARSeq Products Real/Synth Balanced Fine-Tuning

Fine-tunes PARSeq on the balanced Products date recognition dataset. The dataset zip is created locally by `app.scripts.build_products_date_recognition_dataset` and contains PP-OCR-style recognition labels plus cropped date images.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content')
!rm -rf /content/dataset /content/parseq /content/parseq_data /content/output
!mkdir -p /content/dataset /content/output

## Config
Update Drive paths only if your zips live elsewhere.

In [ ]:
from pathlib import Path

DATASET_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/products_date_recognition_balanced.zip')
PARSEQ_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/parseq.zip')

BASE_UNZIP_DIR = Path('/content/dataset')
PARSEQ_DIR = Path('/content/parseq')
PARSEQ_DATA_ROOT = Path('/content/parseq_data')

RUN_NAME = 'smartbite_parseq_products_real_synth_balanced'
OUTPUT_DIR = Path('/content/output') / RUN_NAME
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/models') / RUN_NAME

EPOCHS = 30
BATCH_SIZE = 128
IMG_SIZE = [32, 128]
NUM_WORKERS = 2
DEVICE = 'gpu'  # set to 'cpu' if no GPU is available

assert DATASET_ZIP_DRIVE.exists(), f'Missing dataset zip: {DATASET_ZIP_DRIVE}'
print('DATASET_ZIP_DRIVE =', DATASET_ZIP_DRIVE)

In [ ]:
import shlex
import subprocess
import sys

def run_live(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
run_live(['unzip', '-q', '-o', DATASET_ZIP_DRIVE, '-d', BASE_UNZIP_DIR])

def find_dataset_root(base: Path) -> Path:
    candidates = [p for p in base.rglob('train_label.txt') if (p.parent / 'val_label.txt').exists()]
    assert candidates, 'Could not find dataset root containing train_label.txt and val_label.txt'
    return sorted([p.parent for p in candidates], key=lambda p: len(str(p)))[0]

DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
print('DATASET_ROOT =', DATASET_ROOT)
for name in ['train_label.txt', 'val_label.txt', 'test_label.txt', 'summary.json']:
    p = DATASET_ROOT / name
    print(name, p.exists(), p)
    assert p.exists(), p

## Install PARSeq

In [ ]:
import os
import shutil

os.chdir('/content')
if PARSEQ_DIR.exists():
    shutil.rmtree(PARSEQ_DIR)

if PARSEQ_ZIP_DRIVE.exists():
    run_live(['unzip', '-q', '-o', PARSEQ_ZIP_DRIVE, '-d', '/content'])
    candidates = sorted(Path('/content').glob('parseq*'))
    candidates = [p for p in candidates if (p / 'train.py').exists()]
    assert candidates, 'parseq.zip did not extract to a directory containing train.py'
    extracted = candidates[0]
    if extracted != PARSEQ_DIR:
        extracted.rename(PARSEQ_DIR)
else:
    run_live(['git', 'clone', 'https://github.com/baudm/parseq.git', str(PARSEQ_DIR)], cwd='/content')

print('PARSEQ_DIR =', PARSEQ_DIR)
assert (PARSEQ_DIR / 'train.py').exists()
# train.py imports Hydra directly; install the editable package plus train-time deps.
run_live([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
    'hydra-core', 'lmdb', 'imgaug', 'tensorboardx', 'pytorch-lightning', 'timm'
], cwd=PARSEQ_DIR)

## Convert Labels To PARSeq LMDB

In [ ]:
import lmdb

def write_lmdb(label_file: Path, output_dir: Path) -> int:
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    lines = [line for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    env = lmdb.open(str(output_dir), map_size=1024 * 1024 * 1024 * 4)
    count = 0
    with env.begin(write=True) as txn:
        for line in lines:
            rel_path, text = line.split('	', 1)
            img_path = DATASET_ROOT / rel_path
            if not img_path.exists():
                raise FileNotFoundError(img_path)
            count += 1
            txn.put(f'image-{count:09d}'.encode(), img_path.read_bytes())
            txn.put(f'label-{count:09d}'.encode(), text.encode('utf-8'))
        txn.put('num-samples'.encode(), str(count).encode())
    env.sync(); env.close()
    return count

if PARSEQ_DATA_ROOT.exists():
    shutil.rmtree(PARSEQ_DATA_ROOT)

train_lmdb = PARSEQ_DATA_ROOT / 'train' / 'real' / 'smartbite_products_balanced'
val_lmdb = PARSEQ_DATA_ROOT / 'val' / 'smartbite_products_balanced'
train_n = write_lmdb(DATASET_ROOT / 'train_label.txt', train_lmdb)
val_n = write_lmdb(DATASET_ROOT / 'val_label.txt', val_lmdb)
print('train_lmdb =', train_lmdb, 'samples =', train_n)
print('val_lmdb =', val_lmdb, 'samples =', val_n)
assert train_n > 0 and val_n > 0

## Train PARSeq

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_py = PARSEQ_DIR / 'train.py'
cmd_text = f"""
PYTHONUNBUFFERED=1 HYDRA_FULL_ERROR=1 python -u {train_py} \\
  +experiment=parseq \\
  pretrained=parseq \\
  dataset=real \\
  data.root_dir={PARSEQ_DATA_ROOT} \\
  model.batch_size={BATCH_SIZE} \\
  "model.img_size={IMG_SIZE}" \\
  model.max_label_length=25 \\
  trainer.max_epochs={EPOCHS} \\
  trainer.accelerator={DEVICE} \\
  trainer.devices=1 \\
  trainer.val_check_interval=1.0 \\
  data.num_workers={NUM_WORKERS} \\
  hydra.run.dir={OUTPUT_DIR} \\
  +trainer.enable_progress_bar=True \\
  +trainer.log_every_n_steps=10 \\
  +trainer.num_sanity_val_steps=0 \\
  ++data.augment=False
""".strip()
print('>>', cmd_text)
%cd {PARSEQ_DIR}
!{cmd_text}
print('Training command returned control to notebook. Output dir:', OUTPUT_DIR)

## Inspect, Smoke-Test, And Export

In [ ]:
import torch

ckpts = sorted(OUTPUT_DIR.rglob('*.ckpt'), key=lambda p: p.stat().st_mtime, reverse=True)
print('checkpoints:', [str(p) for p in ckpts[:10]])
assert ckpts, f'No .ckpt files found under {OUTPUT_DIR}'
best_ckpt = next((p for p in ckpts if 'last' not in p.name.lower()), ckpts[0])
print('Selected checkpoint:', best_ckpt)

sample_lines = [line for line in (DATASET_ROOT / 'val_label.txt').read_text(encoding='utf-8').splitlines() if line.strip()][:8]
sample_paths = [DATASET_ROOT / line.split('	', 1)[0] for line in sample_lines]
print('Ground truth:')
for line in sample_lines:
    print(line)
run_live([sys.executable, str(PARSEQ_DIR / 'read.py'), str(best_ckpt), '--device', 'cpu', '--images', *sample_paths], cwd=PARSEQ_DIR)

EXPORT_DIR = OUTPUT_DIR / 'smartbite_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
payload = torch.load(best_ckpt, map_location='cpu')
state_dict = payload.get('state_dict', payload)
torch.save(state_dict, EXPORT_DIR / 'pytorch_model.bin')
(EXPORT_DIR / 'source_checkpoint.txt').write_text(str(best_ckpt), encoding='utf-8')
print('Exported:', EXPORT_DIR / 'pytorch_model.bin')

## Backup Artifacts To Drive

In [ ]:
assert OUTPUT_DIR.exists(), f'Missing output dir: {OUTPUT_DIR}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, FINAL_MODEL_DRIVE_DIR)
print('Saved PARSeq artifacts to:', FINAL_MODEL_DRIVE_DIR)
print('SmartBite export:', FINAL_MODEL_DRIVE_DIR / 'smartbite_export' / 'pytorch_model.bin')